# Task B — real data download and preparation

Downloads the published MetaWIBELE outputs (the baselines we want to beat) and the Nature supplementary tables (positive labels) for the Zhang et al. 2022 IBD paper. Aligns everything to a single set of protein family IDs and saves processed arrays.

**Disk needed: ~2.7 GB.** All downloads are idempotent (skipped if the file already exists). If you want a smaller smoke run, set `SMOKE = True` in the first cell and only the priority + annotation files are downloaded — enough to wire up the pipeline without the 1.5 GB abundance matrix.

**Outputs (to `processed/`):**
- `family_ids.parquet`        — one row per protein family that survives all joins
- `abundance.npy`             — (n_families × n_samples) float32 abundance matrix
- `sample_ids.json`           — column order for `abundance.npy`
- `labels.parquet`            — columns: family_id, is_bioactive (0/1), metawibele_unsup_rank, metawibele_sup_rank
- `sequences.parquet`         — columns: family_id, sequence (amino acids, str)
- `annotations.parquet`       — columns: family_id, characterization (SC/SU/RH/NH), taxonomy (str), ...
- `summary.txt`               — human-readable counts, used to sanity-check downstream notebooks

In [ ]:
from pathlib import Path
import gzip, json, os, shutil, sys
import urllib.request
import pandas as pd
import numpy as np

SMOKE = False  # set True for a minimal download (~210 MB) that still wires up the pipeline

ROOT = Path('.').resolve()
RAW = ROOT / 'raw'
PROC = ROOT / 'processed'
RAW.mkdir(exist_ok=True); PROC.mkdir(exist_ok=True)
print(f'workdir: {ROOT}')
print(f'SMOKE={SMOKE}')

## 1. URLs and file inventory

All MetaWIBELE outputs come from the Huttenhower Lab data portal. Sizes are the published gzip sizes from the page — use them to pick what to skip on a small laptop.

In [ ]:
HUTT = 'http://199.94.60.28/MetaWIBELE_data/HMP2/'  # real download host linked from huttenhower.sph.harvard.edu/metawibele
NATURE = 'https://static-content.springer.com/esm/art%3A10.1038%2Fs41586-022-04648-7/MediaObjects/'

FILES = [
    {'name': 'HMP2_proteinfamilies.centroid.faa.gz',           'url': HUTT + 'HMP2_proteinfamilies.centroid.faa.gz',           'size_mb': 283, 'role': 'sequences (for ESM-2 later)',         'smoke': False},
    {'name': 'HMP2_proteinfamilies.clstr.gz',                   'url': HUTT + 'HMP2_proteinfamilies.clstr.gz',                  'size_mb': 29,  'role': 'cluster -> representative-seq id map','smoke': False},
    {'name': 'HMP2_proteinfamilies_annotation.tsv.gz',          'url': HUTT + 'HMP2_proteinfamilies_annotation.tsv.gz',         'size_mb': 707, 'role': 'characterization, taxonomy, Pfam',     'smoke': False},
    {'name': 'HMP2_proteinfamilies_nrm.tsv.gz',                 'url': HUTT + 'HMP2_proteinfamilies_nrm.tsv.gz',                'size_mb': 1606,'role': 'abundance matrix (families × samples)','smoke': False},
    {'name': 'HMP2_unsupervised_prioritization.rank.table.tsv.gz','url': HUTT + 'HMP2_unsupervised_prioritization.rank.table.tsv.gz','size_mb': 79,  'role': 'baseline 1 (ecology only)',          'smoke': True},
    {'name': 'HMP2_supervised_prioritization.rank.table.tsv.gz', 'url': HUTT + 'HMP2_supervised_prioritization.rank.table.tsv.gz', 'size_mb': 140, 'role': 'baseline 2 (ecology + phenotype)',  'smoke': True},
    # Nature MOESM bundle: Supplementary Tables 2–33 (one XLSX). Table 7 + 8 = MPX-validated positives.
    {'name': '41586_2022_4648_MOESM4_ESM.xlsx',                  'url': NATURE + '41586_2022_4648_MOESM4_ESM.xlsx',              'size_mb': 10,  'role': 'positive labels (Supp Tables 7/8)',  'smoke': True},
]

total = sum(f['size_mb'] for f in FILES if not SMOKE or f['smoke'])
print(f'{"will download" if not SMOKE else "SMOKE mode — will download"}: {total} MB')
for f in FILES:
    skip = SMOKE and not f['smoke']
    marker = 'SKIP' if skip else '→   '
    print(f'  {marker} {f["name"]:60s}  {f["size_mb"]:>5} MB   {f["role"]}')

## 2. Download (idempotent)

Streams to disk with progress every 10 MB. Skips files that already exist. **If a download fails halfway through, delete the partial file from `raw/` and re-run this cell.**

In [ ]:
def download(url: str, dest: Path):
    if dest.exists() and dest.stat().st_size > 0:
        print(f'  exists: {dest.name}  ({dest.stat().st_size/1e6:.1f} MB)')
        return
    tmp = dest.with_suffix(dest.suffix + '.part')
    req = urllib.request.Request(url, headers={'User-Agent': 'thesis-data-fetch/1.0'})
    with urllib.request.urlopen(req) as r, open(tmp, 'wb') as out:
        total = int(r.headers.get('Content-Length', 0))
        seen, chunk = 0, 1 << 20
        next_log = 10 * chunk
        while True:
            buf = r.read(chunk)
            if not buf: break
            out.write(buf); seen += len(buf)
            if seen >= next_log:
                pct = 100 * seen / total if total else 0
                print(f'    {dest.name}: {seen/1e6:>7.1f} MB  ({pct:5.1f}%)')
                next_log += 10 * chunk
    tmp.rename(dest)
    print(f'  done:   {dest.name}  ({dest.stat().st_size/1e6:.1f} MB)')

for f in FILES:
    if SMOKE and not f['smoke']:
        continue
    download(f['url'], RAW / f['name'])

## 3. MetaWIBELE priority scores — the baselines

Both files are one row per protein family with a numeric priority score. The supervised version uses phenotype stats (better-aligned with our task), the unsupervised one uses ecology only.

In [ ]:
# Each priority table is LONG-FORMAT: one row per (familyID, evidence-type[, phenotype-contrast]).
# We only want the meta `priority_score` rows (the paper's headline ranking). For the supervised
# table, every family appears once per phenotype contrast (CD-dysbiosis, UC-dysbiosis); take the
# max rank across contrasts so a family that's strongly prioritised in EITHER disease counts.

def load_priority(path: Path, kind: str) -> pd.DataFrame:
    raw = pd.read_csv(path, sep='\t', compression='gzip',
                      usecols=['familyID', 'evidence', 'rank', 'note'])
    print(f'  {path.name}: raw shape={raw.shape}')
    df = raw[raw['evidence'] == 'priority_score'].copy()
    df['familyID'] = df['familyID'].astype(str)
    df['rank'] = df['rank'].astype('float32')
    if kind == 'sup':
        # rank is in [0,1] with higher = better. Max across (CD,UC) is "best prioritised either way".
        df = df.groupby('familyID', as_index=False)['rank'].max()
    else:
        df = df[['familyID', 'rank']]
    print(f'  -> unique families with priority_score: {len(df):,}')
    return df.rename(columns={'rank': f'metawibele_{kind}_rank'})

prio_uns = load_priority(RAW / 'HMP2_unsupervised_prioritization.rank.table.tsv.gz', 'unsup')
prio_sup = load_priority(RAW / 'HMP2_supervised_prioritization.rank.table.tsv.gz', 'sup')

## 4. Supplementary tables — positive labels

Supplementary Tables 7 and 8 in Zhang et al. 2022 list the MPX (metaproteomics) -validated proteins. These are our positive labels: protein families that were actually detected by mass spectrometry in stool samples, i.e. demonstrably translated and present at non-trivial abundance — the proxy for "bioactive" used by the paper itself.

The MOESM4 XLSX bundles Supplementary Tables 2–33. We list its sheets first, then pick out the ones whose name contains "Table 7" / "Table 8". If Nature changes the file format later (they sometimes do), the sheet listing makes it easy to adjust.

In [ ]:
supp_path = RAW / '41586_2022_4648_MOESM4_ESM.xlsx'
xl = pd.ExcelFile(supp_path)
print(f'sheets ({len(xl.sheet_names)}):')
for s in xl.sheet_names:
    print(f'  - {s}')

def pick_sheet(needle: str) -> str:
    match = [s for s in xl.sheet_names if needle.lower() in s.lower()]
    if not match: raise ValueError(f'no sheet matches {needle!r}; choose from {xl.sheet_names}')
    return match[0]

# Paper labels sheets 'Table S2', 'Table S3', ... — the S is required.
sheet_t7 = pick_sheet('table s7')
sheet_t8 = pick_sheet('table s8')
print(f'\nusing: T7={sheet_t7!r}  T8={sheet_t8!r}')

# Both sheets put the title in row 0 and the real header in row 1.
t7 = pd.read_excel(supp_path, sheet_name=sheet_t7, header=1)
t8 = pd.read_excel(supp_path, sheet_name=sheet_t8, header=1)
print(f'\nT7 shape={t7.shape}, columns={list(t7.columns)}')
print(f'T8 shape={t8.shape}, columns={list(t8.columns)}')

In [ ]:
pos_ids = set()

# Table S7: 'Protein families profiled by metaproteomes in HMP2.' One row per family,
# the `familyID` column lists every family detected by mass spec at least once.
# Optional: keep only the 'MPX-prevalent' subset using the Prevalence column.
PREVALENCE_THRESHOLD = 0.0  # 0 = include any detection; raise (e.g. mean(t7.Prevalence)) for the MPX-prevalent set
t7_keep = t7.dropna(subset=['familyID'])
if PREVALENCE_THRESHOLD > 0 and 'Prevalence' in t7_keep.columns:
    t7_keep = t7_keep[t7_keep['Prevalence'] >= PREVALENCE_THRESHOLD]
t7_ids = t7_keep['familyID'].astype(str).str.strip().unique()
pos_ids.update(t7_ids)
print(f'  T7: kept {len(t7_ids):,} families (threshold={PREVALENCE_THRESHOLD})')

# Table S8: GSEA results. Each row has a slash-separated `core_enrichment` string of family IDs
# that drove the enrichment signal. We union all of them with the T7 set.
t8_ids = set()
if 'core_enrichment' in t8.columns:
    for s in t8['core_enrichment'].dropna().astype(str):
        for fid in s.split('/'):
            fid = fid.strip()
            if fid: t8_ids.add(fid)
    pos_ids.update(t8_ids)
print(f'  T8: kept {len(t8_ids):,} families from core_enrichment')

print(f'\nunion: {len(pos_ids):,} unique positive family ids')

## 5. Annotations — characterization (SC/SU/RH/NH) and taxonomy

Used later to (a) stratify evaluation by characterization category and (b) optionally add as input features.

In [ ]:
# The annotation file is LONG-FORMAT: columns [familyID, annotation, feature, category, method, AID]
# with many rows per family (one per annotation type). For v0 we don't need it as model input,
# but we want a small per-family summary table for stratified eval ("does our model help more
# for uncharacterized families?"). Pivot just two useful axes: homology category (SC/SU/RH/NH)
# and taxonomy lineage.
anno_path = RAW / 'HMP2_proteinfamilies_annotation.tsv.gz'
if anno_path.exists():
    print('streaming annotation file (long format) ...')
    rows = []
    chunk_iter = pd.read_csv(anno_path, sep='\t', compression='gzip',
                             usecols=['familyID', 'annotation', 'feature', 'category', 'method'],
                             chunksize=500_000, low_memory=False)
    for i, ch in enumerate(chunk_iter):
        mask = ch['category'].isin(['UniRef90_homology', 'taxonomy', 'characterization'])
        if mask.any():
            rows.append(ch[mask][['familyID', 'annotation', 'feature', 'category']])
        if i % 5 == 0:
            print(f'  chunk {i}, kept {sum(len(r) for r in rows):,} rows so far')
    anno_long = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    print(f'annotation rows kept: {len(anno_long):,}')

    # Pivot to wide: one row per family with whatever useful columns we found.
    if not anno_long.empty:
        anno = anno_long.drop_duplicates(['familyID', 'feature']).pivot_table(
            index='familyID', columns='feature', values='annotation', aggfunc='first').reset_index()
        anno.columns.name = None
        anno = anno.rename(columns={'familyID': 'family_id'})
        anno['family_id'] = anno['family_id'].astype(str)
        print(f'wide annotation table: {anno.shape}  columns={list(anno.columns)[:10]}')
    else:
        anno = None
else:
    print('annotation file not present (SMOKE mode) — skipping')
    anno = None

## 6. Abundance matrix

1.5 GB gzip, ~1M rows × ~1638 cols. Read in chunks of 10k families to stay under 8 GB RAM. Stored as `float32` numpy. **Skipped in SMOKE mode.**

In [ ]:
abu_path = RAW / 'HMP2_proteinfamilies_nrm.tsv.gz'
if not abu_path.exists():
    print('abundance matrix not present (SMOKE mode) — skipping')
    abundance, family_ids, sample_ids = None, None, None
else:
    print('streaming abundance matrix in chunks...')
    chunks = []
    family_ids = []
    sample_ids = None
    for i, ch in enumerate(pd.read_csv(abu_path, sep='\t', compression='gzip',
                                       chunksize=10_000, low_memory=False)):
        if sample_ids is None:
            id_col = ch.columns[0]
            sample_ids = list(ch.columns[1:])
        family_ids.extend(ch.iloc[:, 0].astype(str).tolist())
        chunks.append(ch.iloc[:, 1:].to_numpy(dtype=np.float32))
        if i % 5 == 0:
            print(f'  chunk {i:3d}  total families = {len(family_ids):>8,}')
    abundance = np.vstack(chunks)
    family_ids = np.array(family_ids, dtype=object)
    print(f'\ndone. abundance: {abundance.shape}  ({abundance.nbytes/1e9:.2f} GB)')
    print(f'samples: {len(sample_ids)}')
    print(f'families: {len(family_ids)}')

## 7. Sequences (FASTA)

One protein sequence per family centroid. Loaded as a `family_id → sequence` dict so we can join on family_id later. Skipped in SMOKE mode.

In [ ]:
fa_path = RAW / 'HMP2_proteinfamilies.centroid.faa.gz'
clstr_path = RAW / 'HMP2_proteinfamilies.clstr.gz'

if not fa_path.exists() or not clstr_path.exists():
    print('FASTA / clstr not present (SMOKE mode) — skipping')
    sequences = None
else:
    # MetaWIBELE FASTA headers are representative-sequence names (e.g. 'CSM79HNM_38926'),
    # NOT cluster IDs ('Cluster_123'). The clstr file maps cluster -> representative seq id.
    # Format is CD-HIT-like:
    #   >Cluster 1
    #   0  150aa, >CSM79HNM_38926... *
    #   1  148aa, >HSM67VID_40426... at 99.33%
    #   ...
    # Only the line ending in '*' is the representative.
    import re
    cluster_to_rep = {}
    cur_cluster = None
    with gzip.open(clstr_path, 'rt') as fh:
        for line in fh:
            if line.startswith('>Cluster'):
                cur_cluster = 'Cluster_' + line.split()[1].strip()
            elif line.rstrip().endswith('*') and cur_cluster is not None:
                m = re.search(r'>([^\s.]+)', line)
                if m: cluster_to_rep[cur_cluster] = m.group(1)
    print(f'cluster -> representative map size: {len(cluster_to_rep):,}')

    raw_seqs = {}
    cur_id, cur_seq = None, []
    with gzip.open(fa_path, 'rt') as fh:
        for line in fh:
            if line.startswith('>'):
                if cur_id is not None: raw_seqs[cur_id] = ''.join(cur_seq)
                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line.strip())
        if cur_id is not None: raw_seqs[cur_id] = ''.join(cur_seq)
    print(f'raw FASTA sequences:                {len(raw_seqs):,}')

    sequences = {cid: raw_seqs[rep] for cid, rep in cluster_to_rep.items() if rep in raw_seqs}
    print(f'cluster -> sequence (joined):       {len(sequences):,}')
    lens = np.array([len(s) for s in sequences.values()])
    print(f'length: min={lens.min()} median={int(np.median(lens))} mean={lens.mean():.0f} max={lens.max()}')

## 8. Join everything on `family_id` and save processed artefacts

In [ ]:
labels = prio_uns.rename(columns={'familyID': 'family_id'}).merge(
    prio_sup.rename(columns={'familyID': 'family_id'}),
    on='family_id', how='outer')
labels['is_bioactive'] = labels['family_id'].isin(pos_ids).astype(np.int8)
print(f'labels: {labels.shape}  positives={int(labels["is_bioactive"].sum()):,} ({labels["is_bioactive"].mean()*100:.3f}%)')

if abundance is not None:
    fa_set = set(family_ids.tolist())
    labels = labels[labels['family_id'].isin(fa_set)].reset_index(drop=True)
    print(f'after restricting to families present in abundance matrix: {len(labels):,} '
          f'(positives={int(labels["is_bioactive"].sum()):,})')

labels.to_parquet(PROC / 'labels.parquet', index=False)
print(f'\nwrote {PROC/"labels.parquet"}')

In [ ]:
if abundance is not None:
    keep_ids = labels['family_id'].values
    fid_to_row = {fid: i for i, fid in enumerate(family_ids)}
    row_idx = np.array([fid_to_row[fid] for fid in keep_ids if fid in fid_to_row])
    print(f'aligning abundance: {len(row_idx):,} rows kept')
    np.save(PROC / 'abundance.npy', abundance[row_idx])
    json.dump(sample_ids, open(PROC / 'sample_ids.json', 'w'))
    pd.DataFrame({'family_id': keep_ids}).to_parquet(PROC / 'family_ids.parquet', index=False)
    print(f'wrote abundance.npy {(PROC/"abundance.npy").stat().st_size/1e6:.1f} MB')

if sequences is not None:
    seq_rows = [{'family_id': fid, 'sequence': sequences.get(fid, '')} for fid in labels['family_id']]
    pd.DataFrame(seq_rows).to_parquet(PROC / 'sequences.parquet', index=False)
    missing = sum(1 for r in seq_rows if not r['sequence'])
    print(f'wrote sequences.parquet ({missing:,} families with empty sequence)')

if anno is not None:
    anno_keep = anno[anno['family_id'].isin(set(labels['family_id']))].copy()
    anno_keep.to_parquet(PROC / 'annotations.parquet', index=False)
    print(f'wrote annotations.parquet  ({len(anno_keep):,} rows kept)')

## 9. Summary

In [ ]:
lines = []
def L(s=''): lines.append(s); print(s)

L('Task B — processed dataset summary')
L('=' * 50)
L(f'families (after all joins):    {len(labels):,}')
L(f'positives (is_bioactive=1):    {int(labels["is_bioactive"].sum()):,}  ({labels["is_bioactive"].mean()*100:.3f}%)')
if abundance is not None:
    L(f'abundance matrix:              {labels.shape[0]:,} × {len(sample_ids):,}  ({(labels.shape[0]*len(sample_ids)*4)/1e9:.2f} GB float32)')
    L(f'samples:                       {len(sample_ids):,}')
if sequences is not None:
    have_seq = sum(1 for fid in labels['family_id'] if sequences.get(fid))
    L(f'sequences present:             {have_seq:,} / {len(labels):,}')
L(f'MetaWIBELE unsup priority:     min={labels["metawibele_unsup_rank"].min():.4f}  max={labels["metawibele_unsup_rank"].max():.4f}')
L(f'MetaWIBELE sup   priority:     min={labels["metawibele_sup_rank"].min():.4f}  max={labels["metawibele_sup_rank"].max():.4f}')

from sklearn.metrics import average_precision_score, roc_auc_score
y = labels['is_bioactive'].astype(int).values

# `rank` is a normalised priority in [0,1] where HIGHER = more bioactive (paper convention).
# Fill missing with 0 (family not in that priority table = not prioritised).
if labels['metawibele_unsup_rank'].notna().any():
    s = labels['metawibele_unsup_rank'].fillna(0).values
    L(f'MetaWIBELE unsup AUPRC = {average_precision_score(y, s):.4f}   AUROC = {roc_auc_score(y, s):.4f}')
if labels['metawibele_sup_rank'].notna().any():
    s = labels['metawibele_sup_rank'].fillna(0).values
    L(f'MetaWIBELE sup   AUPRC = {average_precision_score(y, s):.4f}   AUROC = {roc_auc_score(y, s):.4f}')
L(f'\nrandom baseline AUPRC ≈ base rate ≈ {y.mean():.4f}')
L('\n^ those two MetaWIBELE AUPRCs are the published-method baselines that the DL models need to beat.')
open(PROC / 'summary.txt', 'w').write('\n'.join(lines))

## What's next

- `02_abundance_only.ipynb` reads `processed/abundance.npy` + `processed/labels.parquet` and reuses the smoke-test pipeline on real data. The MetaWIBELE AUPRCs printed above become explicit comparison rows in the results table.
- `03_sequence.ipynb` runs on CSUC: reads `processed/sequences.parquet`, computes ESM-2 embeddings once per family, saves them to `processed/seq_emb.npy`, then trains a fused model.

## Gotchas to watch for when this runs for real

- The Nature MOESM4 sheet headers vary; if `pick_sheet('table 7')` fails, the printed sheet list shows what to substitute.
- Some families in the priority tables won't appear in the abundance matrix (filtered out before publication) and vice versa. The outer-join + restrict-to-abundance step handles this but the final family count will be lower than the 1.6M raw number from the paper.
- Positive labels are extremely imbalanced (~0.05–0.2% expected). Use AUPRC and Precision@K, not AUROC, as the primary metric — AUROC is misleading at this base rate.
- If disk is tight, drop `HMP2_proteinfamilies_nrm.tsv.gz` (1.5 GB) and run sequence-only experiments first on the cluster.